# Data Preparation Deliverable — Space Cadet

This notebook reorganizes the preparation code from `classification_model.ipynb` without modifying it.
It prepares data only: **no estimator is fitted, no cross-validation is run, and no model is evaluated**.

**Methodology clarification, confirmed by the project owner:** the original code targets **10,000 training rows**
by sampling whole objects within target-rate bands. On the current data it produces **10,028 rows from 921 objects**.
Trimming it to exactly 10,000 or drawing a new random train/test split would change the methodology.
We therefore retain the existing time-based `train` / `valid` / `test` assignments, reserve final test first,
and sample only the training pool. The final test has **3,084 reentry observations** and is not part of the training sample.

Read and run from top to bottom with a fresh kernel. Upstream acquisition/build commands are documented below,
not executed automatically. The executable cells read existing inputs and keep preparation outputs in memory;
they do not change input files, contact external services, or save a model.

## 1. Trace the upstream preparation pipeline

These responsibilities stay in their existing scripts instead of being copied into this notebook:

| Stage | Existing implementation | What it actually does |
|---|---|---|
| Catalogue and reentry acquisition | `src/fetch_spacetrack.py`: `credentials`, `login`, `download`, `main` | Authenticates to Space-Track and downloads `satcat.csv` and `decay.csv`; supports a private env file and external raw directory. |
| Orbital-history acquisition | `src/fetch_gp_history_b2.py`: `eligible_ids`, `upload_batch`, `convert_csv_to_parquet` | Uses builder eligibility rules, downloads history for eligible NORAD IDs, converts CSV fields to Parquet strings, uploads batches to private B2, and resumes via batch keys/markers. |
| B2 access | `src/b2_storage.py`: `b2_settings`, `s3_client`, `connect_b2_history` | Supplies boto3 and DuckDB/httpfs access to the private Parquet batches; imports the env reader from `fetch_spacetrack.py`. |
| Column normalization | `src/normalize.py`: alias maps and `normalize` | Maps Space-Track export variants to canonical names and reports required columns; this is header normalization, not statistical scaling. |
| Space-weather cleaning | `src/space_weather.py`: `split_cadence`, `tidy_daily`, `add_trailing_features`, `build` | Separates daily OBS/INT/PRD from monthly PRM, coerces numbers, deduplicates dates, fills/interpolates daily values, scales Kp, and constructs trailing windows. Centered future-looking averages are excluded. |
| Object-level merge | `src/build_dataset.py`: `load_satcat`, `load_decay`, `build_objects` | Deduplicates the catalogue by completeness; collapses decay messages to the latest per object; left-joins on NORAD ID, falls back to catalogue decay dates, derives geometry/lifetime, and clears negative lifetimes. It does **not** restrict messages to Historical. |
| Observation panel | `src/build_dataset.py`: `sample_epochs`, `build_panel` | Keeps terminal perigee ≤ 1,000 km and launches from 1957-10-01; generates 12 lifetime-spaced epochs with seeded jitter; joins object data by ID and weather by observation day. |
| Orbit merge | `attach_orbit_history` / `attach_orbit_history_b2` in the builder | Matches the most recent element with epoch ≤ observation date, derives a trailing perigee slope, and nulls orbit history older than 45 days. B2 uses bounded remote groups and can report skipped incompatible groups. |
| Labels and split | `build_panel` | Derives days remaining, age, censoring duration and log label; assigns dates ≤ the 70th percentile to train, dates through the 85th percentile to valid, and later dates to test. These cutoffs are computed upstream on the full panel, not recomputed after filtering or sampling here. |
| Feature specification | `feature_groups` → `data/processed/feature_groups.json` | Groups orbit, trailing weather, forward weather, outcomes and keys; defines the selected `with_sun_trailing` predictors. |
| Shared settings | `src/config.py` | Defines paths, seed `20261019`, 12 epochs, 1,000-km perigee bound, 45-day staleness and 30-day slope window. |

`SW-All.csv` is obtained separately from CelesTrak as described in the repository README; no inspected Python
script downloads it. `build_dataset.main()` calls `space_weather.build()` **in memory**, then writes
`objects.csv`, `panel.csv`, and `feature_groups.json`. The separate `space_weather.py` CLI writes its standalone
daily/monthly CSVs; those intermediate files are **not inputs** to the main builder or this notebook.

**Evidence and limits of the trace.** A read-only reconstruction from the local raw catalogue, decay and weather
files reproduced the stored object IDs, event flags, terminal altitudes, lifetimes and as-of ages. The configured
epoch generator reproduced every stored panel observation key/epoch index; all split assignments matched the
builder's 70/85-percentile logic, and the current builder reproduced the feature manifest. There is no local
`data/raw/gp_history.csv`; the repository documents and implements the B2 route and the panel has populated
as-of orbit features. The historical command line and versioned B2 batch inventory are not saved in this checkout,
so the precise remote snapshot cannot be independently reconstructed from these files alone. The remote join
was inspected, not rerun or claimed to have been fully reverified.

**Use implemented code as the authority.** `PLAN.md` describes changes not implemented in this checkout,
including calendar snapshots and object-disjoint splits. The README also suggests comparing models on test;
that instruction is **not followed** here: use development training/validation only, then final test once after freezing decisions.

## 2. Environment, input paths and reproducibility

From the repository root, install the existing pipeline dependencies and a notebook kernel:

```powershell
python -m pip install -r requirements.txt
python -m pip install ipykernel nbformat nbclient
```

Open this notebook in VS Code and select that interpreter. The notebook was verified with pandas 2.2.3,
NumPy 2.1.3 and scikit-learn 1.6.1; its runtime versions and input hashes are printed below.
For the same tested core versions, install `pandas==2.2.3 numpy==2.1.3 scikit-learn==1.6.1` in the chosen environment.
The existing requirements use lower bounds, so they alone are not an exact lock. `pyproject.toml` omits
scikit-learn, and `uv.lock`'s project entry omits B2 dependencies added to the current source. This deliverable
uses the requirements-based route and does not silently edit either file.

The default input is the **original local `data/processed/panel.csv`**, not Supabase or its 10,028-row export.
The latter has no validation or final-test rows and cannot reproduce this preparation flow.

In [1]:
from pathlib import Path
import sys
sys.dont_write_bytecode = True
import os
import json
import hashlib
import importlib.metadata
import numpy as np
import pandas as pd
from IPython.display import display

override = os.environ.get('SPACE_CADET_ROOT')
candidates = [Path(override)] if override else [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p.resolve() for p in candidates
                     if (p / 'src/config.py').is_file()
                     and (p / 'data/processed/panel.csv').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open from the space-cadet repository, or set SPACE_CADET_ROOT to its root.')
PANEL_PATH = PROJECT_ROOT / 'data/processed/panel.csv'
MANIFEST_PATH = PROJECT_ROOT / 'data/processed/feature_groups.json'
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from config import RANDOM_SEED
assert RANDOM_SEED == 20261019, 'Review changed seed before reproducing the original sample.'

def file_sha256(path):
    with Path(path).open('rb') as stream:
        digest = hashlib.sha256()
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

input_hashes = {str(p.relative_to(PROJECT_ROOT)): file_sha256(p)
                for p in [PANEL_PATH, MANIFEST_PATH]}
print('Input SHA-256:', json.dumps(input_hashes, indent=2))
print('Versions:', {name: importlib.metadata.version(name)
                    for name in ['pandas', 'numpy', 'scikit-learn']})

Input SHA-256: {
  "data\\processed\\panel.csv": "08fc6d62fda3e4c35684445e4816c4950c913921b5d6c18fed462a714718a5b6",
  "data\\processed\\feature_groups.json": "02f39a3b7f8de27e4855974bf396d72e5dfc79c2d1bb1e64f29a48fcbf5108bd"
}
Versions: {'pandas': '2.2.3', 'numpy': '2.1.3', 'scikit-learn': '1.6.1'}


### Upstream rebuild instructions (manual prerequisite; do not run as part of development)

If the processed inputs are already present, skip acquisition/rebuild. To rebuild with the existing B2 route,
retain the same raw snapshots and remote batch snapshot, configure a **private** env file from `.env.example`,
and run from the repository root. Replace the paths; never submit the credential file.

```powershell
# First obtain the correct SW-All.csv snapshot separately and put it in data/raw/.
python src/fetch_spacetrack.py --raw-dir data/raw --env-file <private-env-file>
python src/fetch_gp_history_b2.py --raw-dir data/raw --env-file <private-env-file>
python src/build_dataset.py --raw-dir data/raw --sw-all data/raw/SW-All.csv --out-dir data/processed --gp-history-source b2 --env-file <private-env-file>
```

Acquisition contacts live services and the history command uploads data. **Do not rerun it just to run this notebook.**
Fetching fresh source data may change the panel, percentile split boundaries and sample despite using the same seed.
If an equivalent archived local `gp_history.csv` is available, the existing `--gp-history-source local --gp-history <path>`
route is an alternative, but has not been verified as identical to the historical B2 snapshot.
The B2 route needs boto3, DuckDB's httpfs extension, network access, Space-Track access for acquisition and B2 credentials.
The cached-panel notebook route needs none of those external connections.

## 3. Load the panel and keep objects with reentry data

Extracted from the original loading cell: retain observations whose `event_observed` flag is 1. This is an
observation-level filter for objects with a recorded reentry, not one row per object. Right-censored observations
are excluded. "Recorded" does not guarantee independently confirmed reentry: the upstream message selection
also admits predictions, as documented in the concerns section. No new rule is introduced here.

In [2]:
panel = pd.read_csv(PANEL_PATH)
required = {'norad_cat_id', 'obs_date', 'split', 'event_observed', 'days_remaining'}
if not required.issubset(panel.columns):
    raise ValueError(f'Missing required columns: {sorted(required - set(panel.columns))}')
if panel['split'].isna().any() or not set(panel['split']).issubset({'train', 'valid', 'test'}):
    raise ValueError('Unexpected or missing upstream split labels.')
source_row_count = len(panel)
training_rows_before_filter = int(panel['split'].eq('train').sum())
reentered_mask = pd.to_numeric(panel['event_observed'], errors='coerce').eq(1)
reentered_data = panel.loc[reentered_mask].copy()
print(f'Original panel: {source_row_count:,} observations, {panel.shape[1]} columns')
print(f'Recorded-reentry observations: {len(reentered_data):,}')

Original panel: 705,440 observations, 81 columns
Recorded-reentry observations: 390,024


## RESERVE FINAL TEST DATA

**Final test is set aside now, before target summaries, feature decisions, missing-value fitting or sampling.**
It is exactly the original notebook's population: `event_observed == 1` and upstream `split == "test"`.
The current snapshot contains **3,084 rows**. Only reservation counts and identity/disjointness checks are allowed here.

The variable `final_test_reserved` is kept raw and **must not be used for training, model selection,
preprocessing decisions, hyperparameter tuning, thresholds, feature selection, or any other model-development decision**.
No target distribution, missingness report, plot, transform, or model score is computed from it in this notebook.
There is deliberately no `X_test`, `y_test`, test-transform call or unlock/evaluation cell.
After every decision is frozen, a separate final evaluation may apply the frozen training-fitted pipeline once.

The existing split is chronological by observation date, **not object-disjoint**; earlier observations of a test
object may belong to training. We preserve that methodology rather than silently removing those objects.
The sample is drawn from training only; validation and final test are not randomly redrawn or resampled.

In [3]:
test_mask = reentered_data['split'].eq('test')
final_test_reserved = reentered_data.loc[test_mask].copy(deep=True)
reserved_test_indices = frozenset(final_test_reserved.index)
reserved_test_count = len(reserved_test_indices)
reentered_data = reentered_data.loc[~test_mask].copy()
assert set(reentered_data['split']).issubset({'train', 'valid'})
assert reserved_test_indices.isdisjoint(reentered_data.index)
assert reserved_test_count > 0
# Remove the full-panel handle so downstream cells only use the development pool.
del panel, reentered_mask, test_mask
print(f'FINAL TEST RESERVED: {reserved_test_count:,} rows; no development use permitted.')
print('Development pool:', reentered_data['split'].value_counts().to_dict())

FINAL TEST RESERVED: 3,084 rows; no development use permitted.
Development pool: {'train': 359744, 'valid': 27196}


## 4. Construct the existing 365-day target on development rows only

Reuse `reenter_within_365d = (days_remaining <= 365)`. Missing or negative target values are errors, never imputed.
Class 0 means **recorded reentry later than 365 days**, not "never reenters". Counts below use training only.

In [4]:
TARGET = "reenter_within_365d"
days_remaining = pd.to_numeric(reentered_data["days_remaining"], errors="coerce")
invalid_target = days_remaining.isna() | days_remaining.lt(0)
if invalid_target.any():
    raise ValueError(
        f"Cannot label {int(invalid_target.sum()):,} rows with missing or negative days_remaining."
    )

reentered_data[TARGET] = days_remaining.le(365).astype("int8")

print('Training target counts (0=later, 1=within 365 days):')
print(reentered_data.loc[reentered_data['split'].eq('train'), TARGET].value_counts().sort_index().to_dict())

Training target counts (0=later, 1=within 365 days):
{0: 204192, 1: 155552}


## 5. Reuse the feature manifest and predictor exclusions

Keep `with_sun_trailing`, the original 52-feature specification. Identifiers, split labels and post-outcome
columns are blocked as predictors. Forward weather is not selected. The terminal `inclination_deg` feature is
still in the original static group; that remaining provenance caveat is documented below, not silently changed.

In [5]:
import json

with (PROJECT_ROOT / "data" / "processed" / "feature_groups.json").open(encoding="utf-8") as f:
    feature_manifest = json.load(f)

FEATURE_SET = "with_sun_trailing"
if FEATURE_SET not in feature_manifest["ablation"]:
    raise ValueError(f"Unknown feature set: {FEATURE_SET}")
feature_columns = feature_manifest["ablation"][FEATURE_SET]

targets = feature_manifest["targets"]
target_columns = [targets["regression"], targets["regression_log"]]
target_columns.extend(targets["survival"].values())
blocked_columns = set(feature_manifest["keys"] + [feature_manifest["split"]])
blocked_columns.update(target_columns)
blocked_columns.update(feature_manifest["leakage_never_use_as_features"])
blocked_columns.add(TARGET)
print("\\nNever use as predictors:", ", ".join(sorted(blocked_columns)))
print(f"\\nSelected feature set: {FEATURE_SET} ({len(feature_columns)} predictors)")
assert not (set(feature_columns) & blocked_columns), 'A blocked outcome or ID appears in the selected features.'

\nNever use as predictors: days_followed, days_remaining, decay_date, decay_pred_spread_days, epoch_index, event_observed, is_decayed, lifetime_days, log1p_days_remaining, n_decay_msgs, norad_cat_id, object_id, object_name, obs_date, reenter_within_365d, split
\nSelected feature set: with_sun_trailing (52 predictors)


## 6. Handle missing values using training data only

This is the original notebook's explicit preparation stage: numeric medians and missing-indicator decisions
come from the **full eligible training split**; categorical gaps receive `__MISSING__`; features entirely missing
in training are dropped. Validation is transformed with the same frozen values, never used to learn them.
Rows and labels are preserved. Final test is not transformed.

**Distinction from the model-training cell:** the original later logistic-regression cell uses raw sampled
predictors and a separate scikit-learn preprocessor fitted inside each training fold. Its manual pandas-filled
tables are not the input to that model. We retain both preparation interfaces explicitly; do not use the globally
filled tables for fold-based model selection, because their medians were learned outside each fold.

In [6]:
if "feature_columns" not in globals() or "FEATURE_SET" not in globals():
    raise RuntimeError("Run the feature-choice cell before preprocessing.")


missing_columns = [column for column in feature_columns if column not in reentered_data.columns]
if missing_columns:
    raise ValueError(f"Panel is missing expected features: {missing_columns}")

if "split" not in reentered_data.columns:
    raise ValueError("The panel is missing its train/valid/test split column.")
train_mask = reentered_data["split"].eq("train")
if not train_mask.any():
    raise ValueError("No re-entered rows are assigned to the training split.")

# Fit all fill values using training rows only.
X_raw = reentered_data.loc[:, feature_columns]
X_train_raw = X_raw.loc[train_mask]
numeric_columns = X_raw.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_columns = [column for column in feature_columns if column not in numeric_columns]
numeric_medians = X_train_raw[numeric_columns].median()
all_missing_numeric = [column for column in numeric_columns if pd.isna(numeric_medians[column])]
numeric_columns = [column for column in numeric_columns if column not in all_missing_numeric]

categorical_fill_values = {}
all_missing_categorical = []
for column in categorical_columns:
    observed_values = X_train_raw[column].dropna()
    if observed_values.empty:
        all_missing_categorical.append(column)
    else:
        categorical_fill_values[column] = "__MISSING__"
categorical_columns = list(categorical_fill_values)
numeric_missing_indicators = [
    column for column in numeric_columns if X_train_raw[column].isna().any()
]

def apply_missing_value_strategy(frame):
    """Apply training-fitted fills without changing the source data."""
    transformed = frame.loc[:, numeric_columns + categorical_columns].copy()
    for column in numeric_missing_indicators:
        transformed[f"{column}__was_missing"] = transformed[column].isna().astype("int8")
    for column in numeric_columns:
        transformed[column] = transformed[column].fillna(numeric_medians[column])
    for column, fill_value in categorical_fill_values.items():
        transformed[column] = transformed[column].astype("string").fillna(fill_value)
    return transformed


X_train = apply_missing_value_strategy(X_raw.loc[reentered_data['split'].eq('train')])
X_valid = apply_missing_value_strategy(X_raw.loc[reentered_data['split'].eq('valid')])
assert reserved_test_indices.isdisjoint(X_train.index)
assert reserved_test_indices.isdisjoint(X_valid.index)
assert not X_train.isna().any().any() and not X_valid.isna().any().any()
print(f'{FEATURE_SET}: {len(feature_columns)} source predictors')
print('Training missing indicators:', len(numeric_missing_indicators))
print('All-missing training features dropped:', all_missing_numeric + all_missing_categorical)
print(f'Prepared training pool: {len(X_train):,}; prepared validation: {len(X_valid):,}')

with_sun_trailing: 52 source predictors
Training missing indicators: 26
All-missing training features dropped: []
Prepared training pool: 359,744; prepared validation: 27,196


## 7. Randomly sample the training pool — target 10,000 rows

The following block is copied from the original sampling code. It uses seed **20261019**, groups training
observations by NORAD ID, bins each object's positive-target rate into the same five bands, allocates each band
a proportional row budget, shuffles its objects, and keeps whole objects until the budget is reached.
This is **not a uniform random draw of exactly 10,000 independent rows**. Whole-object rounding produces
10,028 rows / 921 objects for the inspected snapshot, as agreed. No final-test rows participate in stratification.

In [7]:
train_mask = reentered_data["split"].eq("train")
valid_mask = reentered_data["split"].eq("valid")
train_rows = reentered_data.loc[train_mask, ["norad_cat_id", TARGET]]
sample_target_rows = min(10_000, len(train_rows))
random_seed = 20261019

# Summarize target composition by object, then sample whole objects within rate bands.
group_profile = (
    train_rows.groupby("norad_cat_id", as_index=False)
    .agg(row_count=(TARGET, "size"), positive_rate=(TARGET, "mean"))
)
group_profile["target_rate_band"] = pd.cut(
    group_profile["positive_rate"],
    bins=[-0.001, 0.0, 0.25, 0.5, 0.75, 1.001],
    labels=["0%", "0-25%", "25-50%", "50-75%", "75-100%"],
    include_lowest=True,
)
rng = np.random.default_rng(random_seed)
selected_object_ids = []
for _, band_groups in group_profile.groupby("target_rate_band", observed=True, sort=True):
    band_row_budget = round(sample_target_rows * band_groups["row_count"].sum() / len(train_rows))
    shuffled_groups = band_groups.sample(frac=1, random_state=int(rng.integers(0, 2**32 - 1)))
    rows_selected_in_band = 0
    for group in shuffled_groups.itertuples(index=False):
        if rows_selected_in_band >= band_row_budget:
            break
        selected_object_ids.append(group.norad_cat_id)
        rows_selected_in_band += int(group.row_count)

sample_mask = train_mask & reentered_data["norad_cat_id"].isin(selected_object_ids)
X_train_model = X_raw.loc[sample_mask, feature_columns]
y_train_model = reentered_data.loc[sample_mask, TARGET].astype("int8")
groups_train = reentered_data.loc[sample_mask, "norad_cat_id"]
X_valid_model = X_raw.loc[valid_mask, feature_columns]
y_valid = reentered_data.loc[valid_mask, TARGET].astype("int8")


X_train_sample_prepared = X_train.loc[X_train_model.index].copy()
sampled_training_rows = reentered_data.loc[sample_mask].copy()
assert reserved_test_indices.isdisjoint(X_train_model.index)
assert reserved_test_indices.isdisjoint(X_valid_model.index)
assert set(X_train_model.index).isdisjoint(X_valid_model.index)
assert sampled_training_rows['split'].eq('train').all()
assert set(X_train_model.index) == set(reentered_data.loc[train_mask & reentered_data.norad_cat_id.isin(selected_object_ids)].index)
print(f'Target: {sample_target_rows:,} rows; actual: {len(X_train_model):,} rows from {groups_train.nunique():,} objects')
print(f'Training positive share: {reentered_data.loc[train_mask, TARGET].mean():.3%}; sampled share: {y_train_model.mean():.3%}')
print('Sample label counts:', y_train_model.value_counts().sort_index().to_dict())
print(f'Validation unchanged: {len(X_valid_model):,} rows. Final test remains reserved: {reserved_test_count:,} rows.')

Target: 10,000 rows; actual: 10,028 rows from 921 objects
Training positive share: 43.240%; sampled share: 43.269%
Sample label counts: {0: 5689, 1: 4339}
Validation unchanged: 27,196 rows. Final test remains reserved: 3,084 rows.


## 8. Preserve the model-ready preprocessing definition, without fitting a model

Extract the original scikit-learn preprocessing definition, but omit the classifier, fitting, cross-validation,
predictions and metrics. This **unfitted** `preprocessor` belongs inside the later training pipeline: median
imputation plus indicators and scaling for numeric features, and missing-category imputation plus one-hot
encoding for categorical features. Numeric/categorical roles come from the sampled training predictors only.
Neither scaling nor encoding is learned from validation or final test. No final-test transformation is performed.

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
numeric_features = X_train_model.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = [column for column in feature_columns if column not in numeric_features]
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scale", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

assert not hasattr(preprocessor, 'transformers_'), 'The model-ready preprocessor should remain unfitted here.'
print('Unfitted preprocessing definition ready; no model training or evaluation executed.')

Unfitted preprocessing definition ready; no model training or evaluation executed.


## Identify Data-Quality and Sampling Concerns

The checks below use **development training rows only**. They report issues without adding new cleaning,
resampling, balancing or predictor-selection rules. Final-test features and outcomes are not profiled.

1. **Missing data:** quantify missingness in the original training predictors before imputation, including as-of
   orbit coverage and radar-size gaps. Missing indicators keep availability information. Observed missingness
   patterns alone do not establish MCAR/MAR/MNAR; no missingness mechanism is asserted as proven.
2. **Class balance:** compare training and sampled labels. The observed positive share is about 43%, so describe
   a moderate imbalance rather than calling positives rare. No synthetic balancing is added.
3. **Conditional population and selection:** excluding censored objects changes the population to recorded
   reentries. Terminal-perigee and launch-date eligibility, and tracked-catalogue coverage, further limit generalization.
   The original sampler stratifies whole objects by their training target rates; it is not uniform row sampling.
4. **Outcome-dependent observation times:** upstream epochs are spread between launch and reentry for decayed
   objects, using the known endpoint. Age and sampled epochs can therefore reflect lifetime; these are not neutral
   prospectively scheduled observations. This is retained and disclosed.
5. **Repeated observations:** whole-object sampling keeps training rows together, but the upstream date split
   can put the same object into train and valid/test. Row disjointness is not object disjointness. The later model's
   grouped CV addresses its folds, not the independent upstream split. We do not run that CV here.
6. **Orbit provenance:** terminal orbit fields can leak outcomes; the selected static `inclination_deg` still
   originates from the catalogue's terminal snapshot. The as-of join uses earlier element epochs and a 45-day
   staleness cap, but historical availability at epoch time is not independently proven by these files.
7. **Unusual values:** report negative as-of perigees, reversed altitude bounds, invalid eccentricities, duplicate
   keys, negative ages, and stale/future elements where present. A negative orbital perigee is flagged for domain
   review, not automatically deleted or declared a corrupt record. Zero checks are reported as zero.
8. **Recorded reentry versus confirmed event:** the latest-message collapse accepts Prediction records;
   `build_objects` flags any nonmissing decay date and does not require date ≤ as-of. The independent raw audit
   found 355 objects represented in training whose latest message was Prediction, and 835 training observations
   whose recorded decay date is later than the builder's 2026-10-25 as-of. Those labels need review before claims
   of confirmed reentry; they are not silently relabeled here.
9. **Weather preparation:** daily OBS/INT/PRD values are combined, same-day values are used, and the existing
   two-sided numeric interpolation could borrow later measurements where gaps exist. The inspected snapshot
   has zero reconstructed calendar-gap days; that is not proof that all numeric values were originally present.
   The selected set excludes explicit `fwd_*` and centered-average features. B2 skip/retry logic can also affect
   coverage, but no historical skipped-batch count is invented.
10. **Reproducibility:** stable seeds do not freeze live source revisions, remote B2 batches, library versions or
    dataset row ordering. Preserve the source snapshots and hashes; the current local lockfile is not sufficient.

In [9]:
train_data = reentered_data.loc[train_mask]
train_predictors = X_raw.loc[train_mask]
missing_report = pd.DataFrame({
    'missing_rows': train_predictors.isna().sum(),
    'missing_pct': train_predictors.isna().mean().mul(100).round(3),
}).sort_values('missing_pct', ascending=False)
display(missing_report[missing_report.missing_rows.gt(0)])
print(f'Training rows excluded by the recorded-reentry filter: {training_rows_before_filter-len(train_data):,}')
display(pd.DataFrame({
    'full_training_count': train_data[TARGET].value_counts().sort_index(),
    'sample_count': y_train_model.value_counts().sort_index(),
    'full_training_share': train_data[TARGET].value_counts(normalize=True).sort_index(),
    'sample_share': y_train_model.value_counts(normalize=True).sort_index(),
}))
display(train_data.object_type.value_counts(dropna=False).rename('training_rows').to_frame())

quality_checks = {
    'Duplicate training observation keys': int(train_data.duplicated(['norad_cat_id','obs_date']).sum()),
    'Missing or negative target': int((train_data.days_remaining.isna() | train_data.days_remaining.lt(0)).sum()),
    'Negative object age': int(train_data.age_days.lt(0).sum()),
    'Negative as-of perigee (review)': int(train_data.perigee_km_asof.lt(0).sum()),
    'As-of apogee below perigee': int(train_data.apogee_km_asof.lt(train_data.perigee_km_asof).sum()),
    'As-of eccentricity outside [0,1)': int((train_data.eccentricity_asof.lt(0) | train_data.eccentricity_asof.ge(1)).sum()),
    'Matched element age outside [0,45] days': int((train_data.element_age_days.lt(0) | train_data.element_age_days.gt(45)).sum()),
    'Training rows missing core orbit history': int(train_data.perigee_km_asof.isna().sum()),
}
display(pd.Series(quality_checks, name='training_rows').to_frame())
overlap_train_valid = set(train_data.norad_cat_id) & set(reentered_data.loc[valid_mask, 'norad_cat_id'])
print(f'Objects shared between full training and validation: {len(overlap_train_valid):,}')
by_era = pd.DataFrame({
    'observation_decade': (pd.to_datetime(train_data.obs_date).dt.year // 10) * 10,
    'orbit_history_missing': train_data.perigee_km_asof.isna(),
    'radar_size_missing': train_data.rcs_size.isna(),
}).groupby('observation_decade').agg(rows=('orbit_history_missing','size'),
                                  orbit_missing_share=('orbit_history_missing','mean'),
                                  radar_size_missing_share=('radar_size_missing','mean'))
display(by_era)

,missing_rows,missing_pct
perigee_decay_rate_km_per_day,170713,47.454
eccentricity_asof,163722,45.511
apogee_km_asof,163722,45.511
period_min_asof,163722,45.511
bstar,163722,45.511
mean_motion_dot,163722,45.511
element_age_days,163722,45.511
mean_alt_km_asof,163722,45.511
inclination_deg_asof,163722,45.511
mean_motion,163722,45.511


Training rows excluded by the recorded-reentry filter: 134,084


,full_training_count,sample_count,full_training_share,sample_share
reenter_within_365d,,,,
0,204192,5689,0.567604,0.567312
1,155552,4339,0.432396,0.432688


,training_rows
object_type,
DEBRIS,251452
PAYLOAD,69621
ROCKET BODY,35644
UNKNOWN,3027


,training_rows
Duplicate training observation keys,0
Missing or negative target,0
Negative object age,0
Negative as-of perigee (review),12
As-of apogee below perigee,0
"As-of eccentricity outside [0,1)",0
"Matched element age outside [0,45] days",0
Training rows missing core orbit history,163722


Objects shared between full training and validation: 3,631


,rows,orbit_missing_share,radar_size_missing_share
observation_decade,,,
1950,136,0.639706,0.904412
1960,22296,0.346699,0.898412
1970,51787,0.361539,0.647479
1980,74876,0.489222,0.373725
1990,51444,0.575947,0.155489
2000,64318,0.700379,0.047607
2010,46534,0.415030,0.014763
2020,48353,0.135710,0.004860


### Optional local raw-source audit — no network, no final-test profiling

Reuse the existing cleaning functions to check the available raw snapshots. If raw files are absent, the
cached-panel preparation still runs and this cell reports a skip. These checks do not alter any preparation
decision, fit a model, or read reserved final-test predictors. Mixed ISO date/timestamp strings are parsed for
the diagnostic only; the original modelling inputs are unchanged.

In [10]:
raw_dir = PROJECT_ROOT / 'data/raw'
raw_paths = [raw_dir / name for name in ['SW-All.csv', 'satcat.csv', 'decay.csv']]
if all(p.exists() for p in raw_paths):
    import space_weather
    from build_dataset import load_satcat, load_decay, build_objects
    daily_weather, monthly_weather = space_weather.build(raw_paths[0])
    builder_asof = daily_weather.loc[daily_weather.source.isin(['OBS','INT']), 'date'].max().normalize()
    latest_messages = load_decay(raw_paths[2])
    training_messages = latest_messages[latest_messages.norad_cat_id.isin(train_data.norad_cat_id)]
    predictions = int(training_messages.loc[training_messages.msg_type.eq('Prediction'), 'norad_cat_id'].nunique())
    future_training_dates = int(pd.to_datetime(train_data.decay_date, format='mixed', errors='coerce').gt(builder_asof).sum())
    print('Builder observed-weather as-of:', str(builder_asof.date()))
    print('Training objects with latest Prediction message:', predictions)
    print('Training observations with decay date after builder as-of:', future_training_dates)
    print('Reconstructed weather calendar-gap days:', int(daily_weather.is_gap_filled.sum()))
    print('Raw snapshot hashes:', {p.name: file_sha256(p) for p in raw_paths})
    del daily_weather, monthly_weather, latest_messages, training_messages
else:
    print('Raw-source audit skipped: provide the archived raw files for these optional checks.')

Builder observed-weather as-of: 2026-10-25
Training objects with latest Prediction message: 355
Training observations with decay date after builder as-of: 835
Reconstructed weather calendar-gap days: 0
Raw snapshot hashes: {'SW-All.csv': '01868cd71c9c4c0b64a1aecd7f914b74514592088d06395400e245aff09fbc9f', 'satcat.csv': '195bd18894cf699536071097d2cb7003c84dfff39e836a857e1a40de01729021', 'decay.csv': 'ddfd22edd3cb952d6a65ca100104a2d5ba6a987f27af2542cf8ab62446c119d6'}


## 9. Preparation outputs and final-test guard

| Variable | Intended role |
|---|---|
| `sampled_training_rows` | Original sampled development rows plus the derived 365-day target; current sample is 10,028 rows. |
| `X_train_model`, `y_train_model`, `groups_train` | Raw sampled predictors, labels and object IDs used by the existing model workflow. |
| `X_valid_model`, `y_valid` | Unchanged raw validation predictors and labels; development use only. |
| `X_train_sample_prepared`, `X_valid` | Explicit pandas imputation outputs fitted using full training only, retained to document the original missing-value stage. Do not substitute them for fold-fitted preprocessing during CV. |
| `preprocessor` | Original model-ready scaling/imputation/encoding definition, deliberately **unfitted**. |
| `final_test_reserved` | Untouched raw final-test rows. No development use, no transformations and no evaluation in this notebook. |

No model object, predictions or scores are produced. The original classification notebook is not imported or executed.

In [11]:
assert reserved_test_indices.isdisjoint(sampled_training_rows.index)
assert reserved_test_indices.isdisjoint(X_train.index)
assert reserved_test_indices.isdisjoint(X_valid.index)
assert not hasattr(preprocessor, 'transformers_')
assert not X_train_sample_prepared.isna().any().any()
assert all(file_sha256(PROJECT_ROOT / rel) == digest for rel, digest in input_hashes.items())
print('Preparation complete. Input files unchanged; final test remains reserved.')
print(f'Sampled train: {len(X_train_model):,}; validation: {len(X_valid_model):,}; reserved final test: {reserved_test_count:,}.')

Preparation complete. Input files unchanged; final test remains reserved.
Sampled train: 10,028; validation: 27,196; reserved final test: 3,084.


## 10. Complete preparation-code submission checklist

Preserve these relative paths in the submission. Submit the **current local versions**, including the B2 files
that are untracked in this checkout; a GitHub-only export would omit part of the inspected pipeline.

| Include | Contribution |
|---|---|
| `src/Models/data_preparation.ipynb` | This readable, runnable preparation deliverable, provenance/quality report, reservation policy and execution instructions. |
| `src/config.py` | Shared paths, eligibility limits, windows and reproducibility seed. |
| `src/fetch_spacetrack.py` | Catalogue/decay acquisition and shared credential/env/download helpers. |
| `src/fetch_gp_history_b2.py` | Resumable acquisition and Parquet conversion/upload of eligible orbital histories. |
| `src/b2_storage.py` | B2 settings/client and authenticated DuckDB access required by the B2 build route. |
| `src/normalize.py` | Header aliases and canonical normalization required by builder loaders. |
| `src/space_weather.py` | Source cadence separation, numeric cleaning and weather-feature construction. |
| `src/build_dataset.py` | Object merge, epoch construction, weather/orbit joins, target fields, chronological splits and manifest generation. |
| `data/processed/feature_groups.json` | Exact feature grouping/exclusion specification consumed by the notebook when using the cached panel. |
| `requirements.txt` | Existing dependency list covering the builder, B2 route and scikit-learn preprocessing; notebook-kernel installation is explained above. |
| `.env.example` | Nonsecret Space-Track/B2 environment-variable template; never replace it with a real `.env`. |
| `src/SCHEMA.md` | Field definitions, source keys, units and preparation caveats; read alongside this notebook's observed-code corrections. |

### Inspected/relevant files not needed in the preparation-code submission

| Omit | Reason |
|---|---|
| `src/Models/classification_model.ipynb` | Source of the extracted preparation code; also contains model fitting/evaluation, and is not a runtime dependency of this notebook. Preserve it unchanged locally. |
| `README.md` | Inspected for acquisition/B2 context; this notebook supplies corrected self-contained instructions, while parts of the README are stale or advise test-based comparison. |
| `PLAN.md` | Design/planning record; several listed methods differ from the implemented pipeline and must not be presented as executed preparation. |
| `pyproject.toml`, `uv.lock` | Inspected; the chosen requirements-based route covers the actual imports, while these files are currently inconsistent/incomplete for this pipeline. |
| `src/make_sample_data.py` | Generates explicitly fake smoke-test data, not the real dataset used by the model notebook. |
| `data/raw/SW-All.csv`, `satcat.csv`, `decay.csv` | Data inputs, not preparation code; preserve separately for exact replay and provide only through an approved data-delivery route. |
| B2 `gp_history/batches/*.parquet` and completion sidecars | External data/cache state, not source code; preserve their snapshot/versions separately for a full historical rebuild. |
| `data/processed/panel.csv` | Required cached input for running this notebook, but a large dataset rather than code; deliver separately only if the assignment requires/permits data. |
| `data/processed/objects.csv` | Builder output inspected for provenance; not read by the original classification notebook or required to execute this notebook on a cached panel. |
| Standalone weather CSVs, local `gp_history.csv`, sample fixtures | Not present as runtime inputs to the inspected cached-panel/B2 workflow; weather outputs are regenerable and local GP is an alternative route. |
| Real `.env` files, keys and credentials | Private access material; not inspected for values and never part of the deliverable. |
| Supabase export, dashboard/Modal files, `sample_panel.csv` | Downstream deployment artifacts from a separate workflow; they did not produce the original model notebook's complete panel and splits. |
| `.git`, caches and temporary files | Version-control/runtime state, not preparation source. |

`src/fetch_gp_history.py` is deleted in the inspected working tree; do not substitute a historical version for
the present B2 implementation. No unrelated repository file is changed by this task.

### Dependency and reproducibility boundary

**Cached-panel path:** `data_preparation.ipynb` → `config.py` + `panel.csv` + `feature_groups.json`;
pandas/NumPy for preparation and scikit-learn for the unfitted preprocessing definition. The optional raw audit
also imports `build_dataset.py` → `normalize.py` + `space_weather.py` → `config.py`.

**Raw rebuild path:** Space-Track acquisition → raw catalogue/decay; CelesTrak snapshot → weather builder;
GP fetcher → catalogue/decay loaders + B2 helper + Space-Track helpers → remote Parquet;
dataset builder → normalization/weather/config + B2 helper → objects/panel/feature manifest → this notebook.

The submitted **code is connected and reusable**, but it is **not a self-contained data bundle**. A recipient
needs either the matching archived panel+manifest or the original raw snapshots, B2 history snapshot, dependencies,
and authorized access to rebuild. Seeds alone cannot guarantee a byte-identical rebuild from live services.
No fresh acquisition or full remote-history rebuild was performed during this preparation deliverable.

## Extraction/provenance record

Original notebook source cells (zero-based JSON indices): **1** (load/filter), **3** (target), **5** (feature
manifest and blocked columns), **7** (training-only missing-value rules), and the preparation portion of **9**
(whole-object sampling and unfitted preprocessor definition). Cell **4**'s all-split summaries were replaced by
training-only quality summaries so no final-test distribution is inspected. Classifier imports, fitting,
cross-validation, prediction and metrics were omitted. The source notebook remains unchanged.

The final-test reservation is moved earlier; its transform is deliberately deferred. Full-training pandas
imputation and the later model's raw-input/fold-fitted preprocessor are distinguished instead of conflated.
No new random split, balancing, target definition, eligibility filter or row-count trimming was introduced.

### Inspected snapshot SHA-256

| File | SHA-256 |
|---|---|
| `src\Models\classification_model.ipynb` | `4da4979a5e3496be3eb7a5db6b0cc7d8b907fd0ae792b523c2ebefc7000bca7a` |
| `data\processed\panel.csv` | `08fc6d62fda3e4c35684445e4816c4950c913921b5d6c18fed462a714718a5b6` |
| `data\processed\feature_groups.json` | `02f39a3b7f8de27e4855974bf396d72e5dfc79c2d1bb1e64f29a48fcbf5108bd` |
| `data\raw\SW-All.csv` | `01868cd71c9c4c0b64a1aecd7f914b74514592088d06395400e245aff09fbc9f` |
| `data\raw\satcat.csv` | `195bd18894cf699536071097d2cb7003c84dfff39e836a857e1a40de01729021` |
| `data\raw\decay.csv` | `ddfd22edd3cb952d6a65ca100104a2d5ba6a987f27af2542cf8ab62446c119d6` |